<div align="center">

# Sparse Identification of CRDFT Reactivity Descriptors with SINDy

### Sequentially thresholded least squares on a log-feature library: what sparsity adds, and when

*Supporting computational notebook, companion to* `KAN_from_CRDFT.ipynb` *and* `PySR_from_CRDFT.ipynb`

</div>

---

**Abstract.** SINDy's core regressor, sequentially thresholded least squares (STLSQ), is applied
to the benchmark of the KAN and PySR notebooks on the same 131,375 molecules. The affine
descriptors are fitted on the library $\{IP,EA\}$ and the multiplicative ones on a log-space
**superset** $\{\ln\chi,\ln\eta,\ln A^{+},\ln A^{-}\}$, so sparsity has to *select* which features
enter each law. The notebook also asks what the sparsity step contributes, and reads the KAN and
PySR numbers from their executed notebooks.

## 1. Scope and fairness of the comparison

SINDy fits a target as a sparse linear combination over a fixed library of candidate terms,
$y\approx\Theta(\mathbf{x})\,\xi$, alternating least squares with hard thresholding of small
coefficients. Like the KAN route it reaches the multiplicative CRDFT laws through the logarithm,
but KAN and PySR receive exactly the features a law uses while SINDy receives all four log-features
and has to discard the rest. A closed form counts as recovered only if, after the same rounding
the KAN uses, it is symbolically identical to the reference, with the prefactor taken from
SINDy's own intercept $e^{b}$.

## 2. What STLSQ does here

For a log-space target $\ln y=b+\sum_i \xi_i\ln a_i$, STLSQ solves a lightly ridge-regularized
least-squares problem, zeroes every $|\xi_i|$ below the threshold, re-solves on the surviving
atoms, and repeats until the support is stable. Library and target are centered, so the
intercept is recovered as $b=\bar y-\xi\cdot\bar{\mathbf{x}}$. On a noiseless monomial law the
unthresholded solution is already exact (4.5), so the informative test of sparsity is 4.6.

## 3. Computational methods

Configuration, data loading, the identity check and the anchor are shared with the PySR
notebook. `DRY=1` shrinks the noise study for a quick end-to-end check.

**Cell 3.0. Install and companion modules.** Upload `rsc_style.py`, `figures_rsc.py`,
`extract_kan_results.py`, and either the **executed** `KAN_from_CRDFT.ipynb` or the
`kan_results.json` produced from it. `pysr_results.json` is optional; if present, PySR is added
to the comparison.

In [ ]:
# 3.0 rsc_style.py and figures_rsc.py ship with this repository, not PyPI. The install is
# skipped when the environment already provides everything (clusters build it beforehand).
import os as _os, importlib.util as _ilu
_REQUIRED = ("pysindy", "sklearn", "sympy", "numpy", "pandas", "matplotlib")
_missing = [m for m in _REQUIRED if _ilu.find_spec(m) is None]
if _os.environ.get("SKIP_PIP", "0") == "1" or not _missing:
    print("dependencies already satisfied; skipping pip install"
          + (" (SKIP_PIP=1)" if _os.environ.get("SKIP_PIP") == "1" else ""))
elif _os.path.exists("requirements-sindy.txt"):
    !pip -q install -r requirements-sindy.txt
else:
    !pip -q install pysindy sympy pandas numpy matplotlib

import os, sys
sys.path.insert(0, os.getcwd())
_needed = [f for f in ("rsc_style.py", "figures_rsc.py", "extract_kan_results.py") if not os.path.exists(f)]
if _needed:
    try:
        from google.colab import files
        print("Missing:", ", ".join(_needed)); files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put {', '.join(_needed)} next to this notebook and re-run.")
import importlib, rsc_style, figures_rsc
importlib.reload(rsc_style); importlib.reload(figures_rsc)
# Version guard. Compare as an INTEGER, not a string ("12" < "5" is true).
FIGURES_RSC_MIN = 12
_have = int(getattr(figures_rsc, "__version__", "0"))
if _have < FIGURES_RSC_MIN:
    raise ImportError(
        f"figures_rsc.py is v{_have}; this notebook needs v{FIGURES_RSC_MIN} or newer. "
        "Delete the old file from the file panel (or restart the runtime), upload the "
        "current one, and re-run this cell.")
for _fn in ("fig_sindy_threshold", "fig_method_comparison"):
    if not hasattr(figures_rsc, _fn):
        raise ImportError(f"figures_rsc.py v{_have} has no {_fn}(); upload the current file.")
print(f"companion modules loaded OK (figures_rsc v{figures_rsc.__version__})")

**Cell 3.1. Imports, configuration and environment.**

In [ ]:
import warnings, time, json, hashlib, platform, glob, subprocess
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
from pysindy.optimizers import STLSQ

DRY = os.environ.get("DRY", "0") == "1"
EXPECTED_N = int(os.environ.get("EXPECTED_N", "131375"))   # curated set of the paper (Section 3.1)
THR_AFFINE, THR_LOG, RIDGE = 0.05, 0.15, 1e-6              # STLSQ thresholds and ridge penalty
SIGMAS      = [0.1, 0.2, 0.4]                              # same noise levels as the KAN notebook
NOISE_SEEDS = [0, 7] if DRY else [0, 7, 21, 42, 123]
THRESHOLDS  = [0.01, 0.03, 0.1, 0.3, 0.6, 0.9, 1.1, 1.5, 2.5]
print(("DRY" if DRY else "FULL") + f" configuration | thresholds affine={THR_AFFINE} log={THR_LOG} "
      f"| ridge={RIDGE} | noise sigmas={SIGMAS} x {len(NOISE_SEEDS)} seeds")

import importlib.metadata as _md
def _ver(p):
    try: return _md.version(p)
    except Exception: return "not installed"
print(f"\nEnvironment\n  python   {platform.python_version()}  ({platform.platform()})")
for _p in ["pysindy", "scikit-learn", "sympy", "numpy", "pandas", "matplotlib"]:
    print(f"  {_p:<12} {_ver(_p)}")

**Cell 3.2. Load and curate the dataset.** Identical to the KAN notebook. The SHA-256 of the
CSV is printed so the file can be matched to the archived one, and the notebook **stops** if the
curated molecule count differs from `EXPECTED_N`.

In [ ]:
CSV_PATH = "reactivity_dataset.csv"
USED_SYNTHETIC = not os.path.exists(CSV_PATH)
if not USED_SYNTHETIC:
    DATA_SHA256 = hashlib.sha256(open(CSV_PATH, "rb").read()).hexdigest()
    df = pd.read_csv(CSV_PATH)
else:
    DATA_SHA256 = None
    warnings.warn("reactivity_dataset.csv not found -- using SYNTHETIC data. Results below do "
                  "NOT reproduce the manuscript.", RuntimeWarning)
    print("=" * 74); print("  WARNING: SYNTHETIC DATA IN USE -- results are NOT reproducible"); print("=" * 74)
    rng = np.random.default_rng(7); N = 131385
    IPg = np.clip(rng.normal(8.56, 1.14, N), 3.69, 15.89); EAg = np.clip(rng.normal(-0.46, 1.28, N), -4.45, 4.57)
    etag = IPg - EAg; m = etag > 0.05; IPg, EAg, etag = IPg[m], EAg[m], etag[m]
    mug = -(IPg + EAg) / 2; chig = (IPg + EAg) / 2
    df = pd.DataFrame({"IP": IPg, "EA": EAg, "mu": mug, "chi": chig, "eta": etag, "S": 1 / etag,
                       "Nmax": chig / etag, "omega": mug**2 / (2 * etag),
                       "omega_plus": (IPg + 3 * EAg)**2 / (16 * etag),
                       "omega_minus": (3 * IPg + EAg)**2 / (16 * etag)})

df = df[df["eta"] > 0].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
IP, EA, mu, eta = df.IP.values, df.EA.values, df.mu.values, df.eta.values
chi = -mu; Ap, Am = np.abs(IP + 3 * EA), np.abs(3 * IP + EA)
N_MOL = len(df)
print(f"Molecules: {N_MOL}   (source: {'SYNTHETIC' if USED_SYNTHETIC else CSV_PATH})")
if DATA_SHA256: print(f"CSV sha256: {DATA_SHA256}")
if not USED_SYNTHETIC and N_MOL != EXPECTED_N:
    raise RuntimeError(f"{CSV_PATH} gives {N_MOL} molecules after curation, but the paper's data set "
                       f"has {EXPECTED_N}. This is a different file; fix it (or set EXPECTED_N if intended).")
df[["IP","EA","mu","chi","eta","S","Nmax","omega","omega_plus","omega_minus"]].describe().T[["mean","std","min","max"]].round(4)

**Cell 3.3. Sanity check: the reference identities.**

In [ ]:
checks = {"mu": -(IP+EA)/2, "chi": (IP+EA)/2, "eta": IP-EA, "S": 1/eta, "Nmax": chi/eta,
          "omega": mu**2/(2*eta), "omega_plus": (IP+3*EA)**2/(16*eta), "omega_minus": (3*IP+EA)**2/(16*eta)}
for k, v in checks.items(): print(f"{k:12s} max|res| = {np.abs(v - df[k].values).max():.2e}")

**Cell 3.4. Libraries, reference forms and the STLSQ fitter.**

In [ ]:
IP_s, EA_s, chi_s, eta_s, Ap_s, Am_s = sp.symbols("IP EA chi eta A_plus A_minus")
NAMES = ["mu", "chi", "eta", "S", "Nmax", "omega", "omega_plus", "omega_minus"]
AFFINE, LOGDESC = NAMES[:3], NAMES[3:]
REF = {"mu": -(IP_s+EA_s)/2, "chi": (IP_s+EA_s)/2, "eta": IP_s-EA_s, "S": 1/eta_s,
       "Nmax": chi_s/eta_s, "omega": chi_s**2/(2*eta_s),
       "omega_plus": Ap_s**2/(16*eta_s), "omega_minus": Am_s**2/(16*eta_s)}
# log-feature superset offered to SINDy for every multiplicative descriptor
ATOMS = ["chi", "eta", "Aplus", "Aminus"]
ATOM_LIN = {"chi": chi, "eta": eta, "Aplus": Ap, "Aminus": Am}
ATOM_SYM = {"chi": chi_s, "eta": eta_s, "Aplus": Ap_s, "Aminus": Am_s}
ATOM_TEX = {"chi": r"$\ln\chi$", "eta": r"$\ln\eta$", "Aplus": r"$\ln A^{+}$", "Aminus": r"$\ln A^{-}$"}
TRUE_SUPPORT = {"S": {"eta"}, "Nmax": {"chi", "eta"}, "omega": {"chi", "eta"},
                "omega_plus": {"Aplus", "eta"}, "omega_minus": {"Aminus", "eta"}}
# rows where every atom is non-zero (the log diverges at zero); R2 still uses all rows
G = (Ap > 1e-6) & (Am > 1e-6)
LOGLIB = np.column_stack([np.log(ATOM_LIN[a][G]) for a in ATOMS])

def _snap_half(v): return sp.Rational(round(2*float(v)), 2)
PHYS = [sp.Integer(1), sp.Rational(1,2), sp.Rational(1,4), sp.Rational(1,8), sp.Rational(1,16),
        sp.Integer(2), sp.Integer(3), sp.Integer(16)]
def _snap_const(v):
    c = min(PHYS + [-x for x in PHYS], key=lambda z: abs(float(z) - v))
    return c if abs(float(c) - v) < 0.03*max(abs(v), 1e-9) else sp.nsimplify(v, tolerance=0.01)

def stlsq(X, y, threshold):
    """STLSQ on centred library and target; returns (coefficients, intercept)."""
    mx, my = X.mean(0), y.mean()
    opt = STLSQ(threshold=threshold, alpha=RIDGE)
    opt.fit(X - mx, (y - my).reshape(-1, 1))
    xi = opt.coef_.ravel()
    return xi, float(my - xi @ mx)

def lstsq(X, y):
    A = np.column_stack([X, np.ones(len(y))]); c, *_ = np.linalg.lstsq(A, y, rcond=None)
    return c[:-1], float(c[-1])

def r2_full(expr, col):
    syms = sorted(expr.free_symbols, key=str)
    arr = {IP_s: IP, EA_s: EA, chi_s: chi, eta_s: eta, Ap_s: Ap, Am_s: Am}
    pred = np.broadcast_to(sp.lambdify(syms, expr, "numpy")(*[arr[s] for s in syms]), df[col].shape).astype(float)
    y = df[col].values
    return float(1 - np.var(y - pred) / np.var(y))

def matches(expr, name): return sp.simplify(expr - REF[name]) == 0
def ops(expr): return int(sp.count_ops(expr))

## 4. Results

### 4.1 The exact least-squares anchor (recomputed here)

**Cell 4.1.** Linear least squares on $(IP,EA)$ for the affine descriptors and log-space least
squares on the features for the others, rounded exactly as in the KAN notebook. This is the upper
bound every method is measured against, and its cost.

In [ ]:
TASKS = [("mu", "mu", [IP_s, EA_s], [IP, EA]), ("chi", "chi", [IP_s, EA_s], [IP, EA]),
         ("eta", "eta", [IP_s, EA_s], [IP, EA]), ("S", "S", [eta_s], [eta]),
         ("Nmax", "Nmax", [chi_s, eta_s], [chi, eta]), ("omega", "omega", [chi_s, eta_s], [chi, eta]),
         ("omega_plus", "omega_plus", [Ap_s, eta_s], [Ap, eta]),
         ("omega_minus", "omega_minus", [Am_s, eta_s], [Am, eta])]
anchor = {}
for name, col, syms, arrays in TASKS:
    y = df[col].values
    t0 = time.perf_counter()
    if name in ("mu", "chi", "eta"):
        A = np.column_stack(arrays + [np.ones(N_MOL)])
        c, *_ = np.linalg.lstsq(A, y, rcond=None)
        const = _snap_half(c[-1]) if abs(c[-1]) > 0.05 else sp.Integer(0)
        expr = sp.simplify(sum(_snap_half(ci)*s for ci, s in zip(c[:-1], syms)) + const)
    else:
        g = y > 1e-12
        for a in arrays: g &= np.abs(a) > 1e-9
        A = np.column_stack([np.log(np.abs(a[g])) for a in arrays] + [np.ones(int(g.sum()))])
        c, *_ = np.linalg.lstsq(A, np.log(y[g]), rcond=None)
        expr = _snap_const(float(np.exp(c[-1])))
        for ci, s in zip(c[:-1], syms): expr *= s**_snap_half(ci)
        expr = sp.simplify(expr)
    wall = time.perf_counter() - t0
    anchor[name] = dict(expr=expr, match=matches(expr, name), r2=r2_full(expr, col), wall=wall)
    print(f"{name:12s} {str(expr):24s} match={'yes' if anchor[name]['match'] else 'NO':3s} "
          f"R2={anchor[name]['r2']:.6f}  time={1e3*wall:.1f} ms")

### 4.2 KAN and PySR results (read from their executed notebooks)

**Cell 4.2.** The KAN numbers come from `kan_results.json`, extracted from the executed KAN
notebook by `extract_kan_results.py`; the PySR numbers, if available, from
`pysr_results.json`. Both are rejected if they were produced on a data set of a different size.

In [ ]:
def _find(fname):
    for p in (f"results/{fname}", fname):
        if os.path.exists(p): return p
    return None

kan_path = _find("kan_results.json")
if kan_path is None:
    cands = sorted(glob.glob("KAN_from_CRDFT*.ipynb"), key=os.path.getmtime, reverse=True)
    if not cands:
        try:
            from google.colab import files
            print("Upload the EXECUTED KAN notebook, or kan_results.json:")
            up = list(files.upload())
            cands = [f for f in up if f.endswith(".ipynb")]
            kan_path = _find("kan_results.json")
        except ImportError:
            pass
    if kan_path is None and cands:
        print(f"extracting KAN results from: {cands[0]}")
        r = subprocess.run([sys.executable, "extract_kan_results.py", cands[0]], capture_output=True, text=True)
        print(r.stdout)
        if r.returncode:
            raise RuntimeError("extract_kan_results.py failed -- is that notebook executed?\n" + r.stderr[-800:])
        kan_path = "results/kan_results.json"
if kan_path is None:
    raise FileNotFoundError(f"No KAN results found. Files here: {sorted(glob.glob('*'))}")

def _check_n(res, label):
    if res["n_molecules"] != N_MOL:
        msg = f"{label} results come from {res['n_molecules']} molecules, this notebook has {N_MOL}."
        if USED_SYNTHETIC: warnings.warn(msg + " (tolerated only with synthetic data)")
        else: raise RuntimeError(msg + " They are not comparable.")

kan = json.load(open(kan_path)); _check_n(kan, "KAN")
_L = {"IP": IP_s, "EA": EA_s, "chi": chi_s, "eta": eta_s, "A_plus": Ap_s, "A_minus": Am_s}
KAN = {n: dict(expr=sp.sympify(kan["descriptors"][n]["network_closed_form"], locals=_L),
               wall=kan["descriptors"][n]["wall_s"]) for n in NAMES}
for n in NAMES: KAN[n]["match"] = matches(KAN[n]["expr"], n)
print(f"KAN   ({kan['source_notebook']}, N={kan['n_molecules']}): "
      f"{sum(KAN[n]['match'] for n in NAMES)}/8 recovered by the network alone")

pysr_path = _find("pysr_results.json")
PYSR = None
if pysr_path:
    pysr = json.load(open(pysr_path)); _check_n(pysr, "PySR")
    PYSR = {n: dict(match=pysr["descriptors"][n]["match"], wall=pysr["descriptors"][n]["wall_s"],
                    ops_raw=pysr["descriptors"][n]["ops_raw"]) for n in NAMES}
    print(f"PySR  (N={pysr['n_molecules']}): {sum(PYSR[n]['match'] for n in NAMES)}/8 recovered after pruning")
else:
    print("PySR  results not found -- the comparison below is anchor / KAN / SINDy only.")

### 4.3 SINDy, affine descriptors on $\{IP, EA\}$

**Cell 4.3.** STLSQ with threshold 0.05 on the centered library. Coefficients are rounded to the
nearest half-integer and the intercept is kept only if it exceeds 0.05, as in the anchor.

In [ ]:
SINDY = {}
print(f"{'descriptor':12s} {'xi(IP)':>10s} {'xi(EA)':>10s} {'intercept':>11s}  {'closed form':18s} match {'R2':>9s} {'time':>9s}")
for n in AFFINE:
    t0 = time.perf_counter()
    xi, b = stlsq(np.column_stack([IP, EA]), df[n].values, THR_AFFINE)
    wall = time.perf_counter() - t0
    expr = sp.simplify(_snap_half(xi[0])*IP_s + _snap_half(xi[1])*EA_s + (_snap_half(b) if abs(b) > 0.05 else 0))
    SINDY[n] = dict(expr=expr, match=matches(expr, n), r2=r2_full(expr, n), wall=wall,
                    coef={"IP": float(xi[0]), "EA": float(xi[1])}, intercept=b)
    print(f"{n:12s} {xi[0]:10.6f} {xi[1]:10.6f} {b:11.2e}  {str(expr):18s} {'yes' if SINDY[n]['match'] else 'NO':5s} "
          f"{SINDY[n]['r2']:9.6f} {1e3*wall:7.1f} ms")

### 4.4 SINDy, multiplicative descriptors on the log-feature superset

**Cell 4.4.** STLSQ with threshold 0.15 on $\{\ln\chi,\ln\eta,\ln A^{+},\ln A^{-}\}$ for every
multiplicative descriptor. All four coefficients are shown, so the selected support is visible;
the prefactor is SINDy's own $e^{b}$, rounded to the same library as the KAN.

In [ ]:
print(f"{'descriptor':12s} " + " ".join(f"{'xi('+a+')':>11s}" for a in ATOMS) +
      f" {'e^b':>9s}  {'support ok':10s} {'closed form':22s} match {'time':>8s}")
for n in LOGDESC:
    y = df[n].values; g = G & (y > 1e-12)
    X = np.column_stack([np.log(ATOM_LIN[a][g]) for a in ATOMS])
    t0 = time.perf_counter()
    xi, b = stlsq(X, np.log(y[g]), THR_LOG)
    wall = time.perf_counter() - t0
    support = {a for a, v in zip(ATOMS, xi) if v != 0}
    expr = _snap_const(float(np.exp(b)))
    for a, v in zip(ATOMS, xi):
        if v != 0: expr *= ATOM_SYM[a]**_snap_half(v)
    expr = sp.simplify(expr)
    SINDY[n] = dict(expr=expr, match=matches(expr, n), r2=r2_full(expr, n), wall=wall,
                    coef={a: float(v) for a, v in zip(ATOMS, xi)}, prefactor=float(np.exp(b)),
                    support_ok=(support == TRUE_SUPPORT[n]))
    print(f"{n:12s} " + " ".join(f"{v:11.6f}" for v in xi) + f" {np.exp(b):9.6f}  "
          f"{'yes' if SINDY[n]['support_ok'] else 'NO':10s} {str(expr):22s} {'yes' if SINDY[n]['match'] else 'NO':5s} "
          f"{1e3*wall:6.1f} ms")
n_sindy = sum(SINDY[n]["match"] for n in NAMES)
print(f"\nSINDy recovers {n_sindy}/8 closed forms; R2 on all molecules: "
      + ", ".join(f"{n} {SINDY[n]['r2']:.6f}" for n in NAMES))

### 4.5 What does sparsity add on exact data?

**Cell 4.5.** The same superset, fitted by plain least squares with no thresholding. If the
coefficients of the irrelevant features are already at round-off level, the support is determined
by least squares alone and the threshold has nothing to remove.

In [ ]:
print(f"{'descriptor':12s} " + " ".join(f"{'LS xi('+a+')':>15s}" for a in ATOMS) + f" {'max |off-support|':>18s}")
LS_EXACT = {}
for n in LOGDESC:
    y = df[n].values; g = G & (y > 1e-12)
    X = np.column_stack([np.log(ATOM_LIN[a][g]) for a in ATOMS])
    c, _ = lstsq(X, np.log(y[g]))
    off = max(abs(v) for a, v in zip(ATOMS, c) if a not in TRUE_SUPPORT[n])
    LS_EXACT[n] = off
    print(f"{n:12s} " + " ".join(f"{v:15.2e}" for v in c) + f" {off:18.2e}")
print(f"\nLargest coefficient on an irrelevant atom, plain least squares: {max(LS_EXACT.values()):.1e}")

### 4.6 Sparsity under multiplicative noise

**Cell 4.6.** The noise model of the KAN notebook, $y\to y\,e^{\varepsilon}$ with
$\varepsilon\sim\mathcal N(0,\sigma)$, for every multiplicative descriptor and noise seed. For
plain least squares the largest weight on an irrelevant atom is recorded; for STLSQ, whether the
selected support is exactly right and the largest exponent error on it.

In [ ]:
rows_noise = []
for sg in SIGMAS:
    for n in LOGDESC:
        y = df[n].values; g = G & (y > 1e-12)
        X = np.column_stack([np.log(ATOM_LIN[a][g]) for a in ATOMS]); ly = np.log(y[g])
        true = {"S": {"eta": -1}, "Nmax": {"chi": 1, "eta": -1}, "omega": {"chi": 2, "eta": -1},
                "omega_plus": {"Aplus": 2, "eta": -1}, "omega_minus": {"Aminus": 2, "eta": -1}}[n]
        ls_off, st_ok, st_err, ls_err = [], [], [], []
        for sd in NOISE_SEEDS:
            # independent stream per (seed, sigma), never shared with the data generator
            yn = ly + np.random.default_rng([sd, int(1000*sg), 2026]).normal(0.0, sg, ly.size)
            c, _ = lstsq(X, yn)
            ls_off.append(max(abs(v) for a, v in zip(ATOMS, c) if a not in true))
            ls_err.append(max(abs(c[ATOMS.index(a)] - p) for a, p in true.items()))
            xi, _ = stlsq(X, yn, THR_LOG)
            st_ok.append({a for a, v in zip(ATOMS, xi) if v != 0} == set(true))
            st_err.append(max(abs(xi[ATOMS.index(a)] - p) for a, p in true.items()))
        rows_noise.append(dict(sigma=sg, name=n, ls_off=float(np.mean(ls_off)), ls_err=float(np.mean(ls_err)),
                               st_support=f"{sum(st_ok)}/{len(st_ok)}", st_err=float(np.mean(st_err))))
noise = pd.DataFrame(rows_noise)
print("mean over seeds: LS = plain least squares, ST = STLSQ; err = largest |exponent error| on the true atoms")
display(noise.pivot(index="name", columns="sigma", values=["ls_off", "ls_err", "st_support", "st_err"])
             .reindex(LOGDESC).round(4))

### 4.7 Threshold sweep

**Cell 4.7 (Figure C2).** Coefficients of all four features against the STLSQ threshold, on the
exact data. The sweep goes past the smallest true exponent ($|{-1}|$) to show where the support
breaks, and the table gives the range of thresholds returning exactly the right features.

In [ ]:
SWEEP = {}
ok_range = {}
for n in LOGDESC:
    y = df[n].values; g = G & (y > 1e-12)
    X = np.column_stack([np.log(ATOM_LIN[a][g]) for a in ATOMS]); ly = np.log(y[g])
    SWEEP[n] = {a: [] for a in ATOMS}; good = []
    for th in THRESHOLDS:
        xi, _ = stlsq(X, ly, th)
        for a, v in zip(ATOMS, xi): SWEEP[n][a].append(float(v))
        if {a for a, v in zip(ATOMS, xi) if v != 0} == TRUE_SUPPORT[n]: good.append(th)
    ok_range[n] = (min(good), max(good)) if good else None
    print(f"{n:12s} correct support for thresholds {ok_range[n][0]} ... {ok_range[n][1]}" if good
          else f"{n:12s} correct support at no threshold tested")

import rsc_style as rsc, figures_rsc as F
rsc.use()
F.fig_sindy_threshold({k: SWEEP[k] for k in ("omega", "omega_plus")}, THRESHOLDS, ATOM_TEX,
                      stem="figC2_sindy_threshold");

### 4.8 Head-to-head comparison

**Cell 4.8 (Table and Figure C3).** Anchor, KAN (network alone), PySR (if available) and SINDy
side by side, with the cost of each fit. Written to `results/sindy_results.json`.

In [ ]:
methods = ["anchor", "KAN"] + (["PySR"] if PYSR else []) + ["SINDy"]
rows = []
for n in NAMES:
    r = dict(name=n, ops_canonical=ops(REF[n]), anchor=anchor[n]["match"], KAN=KAN[n]["match"],
             SINDy=SINDY[n]["match"], wall_anchor=anchor[n]["wall"], wall_KAN=KAN[n]["wall"],
             wall_SINDy=SINDY[n]["wall"])
    if PYSR: r.update(PySR=PYSR[n]["match"], wall_PySR=PYSR[n]["wall"], ops_raw_PySR=PYSR[n]["ops_raw"])
    rows.append(r)
tab = pd.DataFrame(rows).set_index("name")
show = tab[[m for m in methods] + [f"wall_{m}" for m in methods]].copy()
for m in methods:
    w = show[f"wall_{m}"]
    show[f"wall_{m}"] = [f"{1e3*v:.1f} ms" if v < 1 else f"{v:.1f} s" for v in w]
display(show)

os.makedirs("results", exist_ok=True)
json.dump(dict(n_molecules=N_MOL, data_sha256=DATA_SHA256, synthetic=USED_SYNTHETIC,
               config=dict(threshold_affine=THR_AFFINE, threshold_log=THR_LOG, ridge=RIDGE,
                           sigmas=SIGMAS, noise_seeds=NOISE_SEEDS),
               descriptors={n: dict(closed_form=str(SINDY[n]["expr"]), match=bool(SINDY[n]["match"]),
                                    r2=SINDY[n]["r2"], wall_s=SINDY[n]["wall"], coef=SINDY[n]["coef"],
                                    support_ok=SINDY[n].get("support_ok"), prefactor=SINDY[n].get("prefactor"),
                                    ls_offsupport_exact=LS_EXACT.get(n),
                                    threshold_range=ok_range.get(n)) for n in NAMES},
               noise=rows_noise),
          open("results/sindy_results.json", "w"), indent=2)
print("wrote results/sindy_results.json")
F.fig_method_comparison(rows, methods=tuple(methods), stem="figC3_method_comparison");